# Basic processing of ECG and PPG 

Import libraries

In [4]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import neurokit2 as nk

plt.style.use("seaborn-v0_8-whitegrid")

In [5]:
from pathlib import Path

import seaborn as sns

mpl.use("Qt5Agg")
plt.style.use("seaborn-v0_8-whitegrid")  # Use seaborn style for better aesthetics

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

import sys
PROCESSING = Path.cwd() if (Path.cwd() / "notebook_paths.py").exists() else Path.cwd() / "offline_processing"
if str(PROCESSING.resolve()) not in sys.path:
    sys.path.insert(0, str(PROCESSING.resolve()))

from notebook_paths import input_path

VERITY_SENSE_PPG_PATH = Path(input_path("verity_sense_path"), "ppg/")
VERITY_SENSE_ACC_PATH = Path(input_path("verity_sense_path"), "acc/")

verity_sense_ppg_files = sorted(list(VERITY_SENSE_PPG_PATH.glob("*.csv")))
verity_sense_acc_files = sorted(list(VERITY_SENSE_ACC_PATH.glob("*.csv")))

assert VERITY_SENSE_PPG_PATH.exists(), f"FIT file not found: {VERITY_SENSE_PPG_PATH.resolve()}"
VERITY_SENSE_PPG_PATH.resolve()

WindowsPath('C:/Users/marce/Documents/garmin/sleep/verity_sense/2026-09-27/ppg')

## Read and concatenate VeritySense files

In [6]:
ppg = pd.DataFrame()
for i, f in enumerate(verity_sense_ppg_files):
    print(f"{i}: {f.name}")
    # read the CSV file into a DataFrame
    df = pd.read_csv(f, index_col=0)
    df.index = pd.to_datetime(df.index, origin=pd.Timestamp("2000-01-01"),unit = "ns")
    # df.columns = [f"PPG_{i}" for i in range(df.shape[1])]
    ppg = pd.concat([ppg, df], axis=0)

ppg.columns = [f"PPG_{i}" for i in range(ppg.shape[1])]
ppg.rename(columns={ppg.columns[-1]: "Ambient"}, inplace=True)

# plt.figure(figsize=(12, 6))
# plt.subplot(2, 1, 1)
# plt.plot(ppg.index, label="PPG Signal")
# plt.subplot(2, 1, 2, sharex = plt.subplot(2, 1, 1))
# plt.plot(np.diff(ppg.index.values) / np.timedelta64(1, 's'), label="Time Difference (s)")
# plt.ylabel("Time Difference (s)")
# plt.show()

acc = pd.DataFrame()
for i, f in enumerate(verity_sense_acc_files):
    print(f"{i}: {f.name}")
    # read the CSV file into a DataFrame
    df = pd.read_csv(f, index_col=0)
    df.index = pd.to_datetime(df.index, origin=pd.Timestamp("2000-01-01"),unit = "ns")
    # df.columns = [f"ACC_{i}" for i in range(df.shape[1])]
    acc = pd.concat([acc, df], axis=0)

acc.columns = [["x", "y", "z"]]
acc

0: verity_ppg_20260927_18Z_recovery1_000049.csv
1: verity_ppg_20260927_19Z_recovery1_000051.csv
2: verity_ppg_20260927_20Z_recovery1_000055.csv
3: verity_ppg_20260927_21Z_recovery1_000062.csv
4: verity_ppg_20260927_22Z_recovery1_000063.csv
5: verity_ppg_20260927_23Z_recovery1_000070.csv
6: verity_ppg_20260928_00Z_recovery1_000071.csv
7: verity_ppg_20260928_01Z_recovery1_000077.csv
8: verity_ppg_20260928_02Z_recovery1_000081.csv
9: verity_ppg_20260928_03Z_recovery1_000084.csv
10: verity_ppg_20260928_04Z_recovery1_000089.csv
11: verity_ppg_20260928_05Z_recovery1_000091.csv
12: verity_ppg_20260928_06Z_recovery1_000097.csv
0: verity_acc_20260927_18Z_recovery1_000050.csv
1: verity_acc_20260927_19Z_recovery1_000054.csv
2: verity_acc_20260927_20Z_recovery1_000058.csv
3: verity_acc_20260927_21Z_recovery1_000059.csv
4: verity_acc_20260927_22Z_recovery1_000066.csv
5: verity_acc_20260927_23Z_recovery1_000067.csv
6: verity_acc_20260928_00Z_recovery1_000072.csv
7: verity_acc_20260928_01Z_recovery1_

,x,y,z
sensor_timestamp_ns,,,
2017-01-04 20:07:46.777364416,-137,-729,-699
2017-01-04 20:07:46.796595200,-141,-728,-695
2017-01-04 20:07:46.815825984,-147,-729,-700
2017-01-04 20:07:46.835056704,-146,-724,-695
2017-01-04 20:07:46.854287488,-143,-725,-697
...,...,...,...
2017-01-05 08:35:52.911976104,-32,-995,143
2017-01-05 08:35:52.930438016,-8,-1004,142
2017-01-05 08:35:52.948899904,-22,-996,158


In [49]:
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(ppg[["PPG_0", "PPG_1", "PPG_2"]])
plt.title("PPG Signals")
plt.ylabel("Amplitude")
plt.legend(["PPG_0", "PPG_1", "PPG_2"], loc = "upper right")
plt.subplot(2, 1, 2, sharex=plt.subplot(2, 1, 1))
plt.plot(ppg["Ambient"])
plt.title("Ambient Light")
plt.ylabel("Amplitude")
plt.show()

In [7]:
fs_ecg = 130
fs_ppg = 55
fs_acc = 52

ppg = ppg["PPG_0"]  # Use only the first PPG channel for analysis

### Filter PPG

Indeed, the PPG shows some weird low frequency oscillations, as well as high frequency noise. Let's look at the signal in the frequency domain

In [54]:
psd_welch = nk.signal_psd(ppg.values[41000:49040], sampling_rate = fs_ppg, method="welch", min_frequency = 0.05)
plt.figure(figsize = (11,5))
plt.plot(psd_welch["Frequency"], psd_welch["Power"], linewidth = 1.9)
plt.xlabel("Frequency (Hz)", fontsize = 18)
plt.xticks(fontsize = 18)
plt.title("Raw PPG Power Spectral Density", fontsize = 21)
plt.xlim(0,5)
plt.show()

Therefore, we clean the PPG with a band-pass filter (low-pass that removes the high frequency noise + high pass that removes low frequency oscillations). We'll use the `nk.signal_filter()` function of Neurokit

In [8]:
# Band-pass between 0.5 and 8 Hz (Elgendi2013) 
ppg_filtered = -nk.signal_filter(signal = ppg.values, sampling_rate = fs_ppg, lowcut = 0.5, highcut = 8,
                            order = 3, method = "butterworth")

### Extract systolic feet from PPG

We'll use the Aboy++ algorithm. Why? It was recently benchmarked as one of the best systolic feet detectors in PPG signals. See:
- pyPPG: a Python toolbox for comprehensive photoplethysmography signal analysis. Goda et. al, 2024
- Detecting beats in the photoplethysmogram: benchmarking open-source algorithms. Charlton et. al, 2023

The algorithm is implemented in the function *ppg_peak_onset()* inside the script *ppg_fiducials.py*. As long as the script is inside the working directory, we can import the function with the line 

```from *script_name* import *function_name*```

In [9]:
import numpy as np
from dotmap import DotMap
from scipy.signal import kaiserord, firwin, filtfilt, detrend, periodogram, lfilter, find_peaks, firls, resample
from scipy import signal
import copy

def get_peak_onset(ppg, fs, peak_detector='PPGdet'):
        '''PPGdet detects beats in a photoplethysmogram (PPG) signal
        using the improved 'Automatic Beat Detection' of Aboy M et al.

        '''

        # inputs
        x = copy.deepcopy(ppg)                    #signal
        fso=fs
        fs = 75
        x = resample(x, int(len(ppg)*(fs/fso)))
        up = set_beat_detection()                 #settings
        win_sec=10
        w = fs * win_sec                                    #window length(number of samples)
        win_starts = np.array(list(range(0,len(x),round(0.8*w))))
        win_starts = win_starts[0:min(np.where([win_starts >= len(x) - w])[1])]
        win_starts = np.insert(win_starts,len(win_starts), len(x) + 1 - w)

        # before pre-processing
        hr_win=0  #the estimated systolic peak-to-peak distance, initially it is 0
        hr_win_v=[]
        px = detect_maxima(x, 0, hr_win, peak_detector) # detect all maxima
        if len(px)==0:
            peaks = []
            onsets = []
            return peaks, onsets

        # detect peaks in windows
        all_p4 = []
        all_hr = np.empty(len(win_starts)-1)
        all_hr [:] = np.nan
        hr_past = 0 # the actual heart rate
        hrvi = 0    # heart rate variability index

        for win_no in range(0,len(win_starts) - 1):
            curr_els = range(win_starts[win_no],win_starts[win_no] + w)
            curr_x = x[curr_els]

            y1 = def_bandpass(sig = curr_x, fs = fs, lower_cutoff = 0.9 * 30/60, upper_cutoff = 3 * 200/60)   # Filter no.1
            hr = estimate_HR(y1, fs, up, hr_past)               # Estimate HR from weakly filtered signal
            hr_past=hr
            all_hr[win_no] = hr

            if (peak_detector=='PPGdet') and (hr>40):
                if win_no==0:
                    p1 = detect_maxima(y1, 0, hr_win, peak_detector)
                    tr = np.percentile(np.diff(p1), 50)
                    pks_diff = np.diff(p1)
                    pks_diff = pks_diff[pks_diff>=tr]
                    hrvi = np.std(pks_diff) / np.mean(pks_diff) * 5

                hr_win = fs / ((1 + hrvi) * 3)
                hr_win_v.append(hr_win)
            else:
                hr_win=0

            y2 = def_bandpass(curr_x, fs, 0.9 * up.fl_hz, 2.5 * hr / 60)           # Filter no. 2
            y2_deriv = estimate_deriv(y2)                                          # Estimate derivative from highly filtered signal
            p2 = detect_maxima(y2_deriv, up.deriv_threshold,hr_win, peak_detector) # Detect maxima in derivative
            y3 = def_bandpass(curr_x, fs, 0.9 * up.fl_hz, 10 * hr / 60)
            p3 = detect_maxima(y3, 50, hr_win, peak_detector)                      # Detect maxima in moderately filtered signal
            p4 = find_pulse_peaks(p2, p3)
            p4 = np.unique(p4)

            if peak_detector=='PPGdet':
                if len(p4)>round(win_sec/2):
                    pks_diff = np.diff(p4)
                    tr = np.percentile(pks_diff, 30)
                    pks_diff = pks_diff[pks_diff >= tr]

                    med_hr=np.median(all_hr[np.where(all_hr>0)])
                    if ((med_hr*0.5<np.mean(pks_diff)) and (med_hr*1.5<np.mean(pks_diff))):
                        hrvi = np.std(pks_diff) / np.mean(pks_diff)*10

            all_p4 = np.concatenate((all_p4, win_starts[win_no] + p4), axis=None)

        all_p4=all_p4.astype(int)
        all_p4 = np.unique(all_p4)

        peaks = (all_p4/fs*fso).astype(int)
        onsets, peaks = find_onsets(ppg, fso, up, peaks,60/np.median(all_hr)*fs)
        return peaks, onsets




def detect_maxima(sig: np.array, percentile: int ,hr_win: int, peak_detector: str):
        #Table VI pseudocode
        """
        Detect Maxima function detects all peaks in the raw and also in the filtered signal to find.

        :param sig: array of signal with shape (N,) where N is the length of the signal
        :type sig: 1-d array
        :param percentile: in each signal partition, a rank filter detects the peaks above a given percentile
        :type percentile: int
        :param hr_win: window for adaptive the heart rate estimate
        :type hr_win: int
        :param peak_detector: type of peak detector
        :type peak_detector: str

        :return: maximum peaks of signal, 1-d array.

        """

        tr = np.percentile(sig, percentile)

        if peak_detector=='ABD':

            s1,s2,s3 = sig[2:], sig[1:-1],sig[0:-2]
            m = 1 + np.array(np.where((s1 < s2) & (s3 < s2)))
            max_pks = m[sig[m] > tr]

        if peak_detector=='PPGdet':
            s1,s2,s3 = sig[2:], sig[1:-1],sig[0:-2]

            max_loc = []
            min_loc = []
            max_pks=[]
            intensity_v = []
            if hr_win == 0:
                m = 1 + np.array(np.where((s1 < s2) & (s3 < s2)))
                max_pks = m[sig[m] > tr]
            else:
                max_loc = find_peaks(sig, distance=hr_win)[0]
                min_loc = find_peaks(-sig, distance=hr_win)[0]

                for i in range(0,len(max_loc)):
                    values = abs(max_loc[i] - min_loc)
                    min_v = min(values)
                    min_i = np.where(min_v==values)[0][0]
                    intensity_v.append(sig[max_loc[i]] - sig[min_loc[min_i]])

                # possible improvements:
                #   - using adaptive thresholding for the maximum
                #   - estimate probability density of the maximum

                tr2 = np.mean(intensity_v)*0.25
                max_pks = find_peaks(sig+min(sig),prominence=tr2,distance=hr_win)[0]

        return max_pks

def find_pulse_peaks(p2: np.array, p3: np.array):
        """
        Pulse detection function detect the pulse peaks according to the peaks of 1st and 2nd derivatives
        General least-squares smoothing and differentiation by the convolution (Savitzky Golay) method

        :param p2: peaks of the 1st derivatives
        :type p2: 1-d array
        :param p3: peaks of the 2nd derivatives
        :type p2: 1-d array

        :return: pulse peaks, 1-d array.

        """

        p4 = np.empty(len(p2))
        p4[:] = np.nan
        for k in range(0,len(p2)):
            rel_el = np.where(p3>p2[k])
            if np.any(rel_el) and ~np.isnan(rel_el[0][0]):
                p4[k] = p3[rel_el[0][0]]

        p4 = p4[np.where(~np.isnan(p4))]
        p4 = p4.astype(int)
        return p4

def set_beat_detection():
        """
        This function setups the filter parameters of the algorithm

        :return: filter parameters of the algorithm, DotMap

        """
        # plausible HR limits
        up=DotMap()
        up.fl = 30               #lower bound for HR
        up.fh = 200              #upper bound for HR
        up.fl_hz = up.fl/60
        up.fh_hz = up.fh/60

        # Thresholds
        up.deriv_threshold = 75          #originally 90
        up.upper_hr_thresh_prop = 2.25   #originally 1.75
        up.lower_hr_thresh_prop = 0.5    #originally 0.75

        # Other parameters
        up.win_size = 10    #in secs

        return up

def def_bandpass(sig: np.array, fs: int, lower_cutoff: float, upper_cutoff: float):
        """
        def_bandpass filter function detects all peaks in the raw and also in the filtered signal to find.

        :param sig: array of signal with shape (N,) where N is the length of the signal
        :type sig: 1-d array
        :param fs: sampling frequency
        :type fs: int
        :param lower_cutoff: lower cutoff frequency
        :type lower_cutoff: float
        :param upper_cutoff: upper cutoff frequency
        :type upper_cutoff: float

        :return: bandpass filtered signal, 1-d array.

        """

        # Filter characteristics: Eliminate VLFs (below resp freqs): For 4bpm cutoff
        up = DotMap()
        up.paramSet.elim_vlf.Fpass = 1.3*lower_cutoff   #in Hz
        up.paramSet.elim_vlf.Fstop = 0.8*lower_cutoff   #in Hz
        up.paramSet.elim_vlf.Dpass = 0.05
        up.paramSet.elim_vlf.Dstop = 0.01

        # Filter characteristics: Eliminate VHFs (above frequency content of signals)
        up.paramSet.elim_vhf.Fpass = 1.2*upper_cutoff   #in Hz
        up.paramSet.elim_vhf.Fstop = 0.8*upper_cutoff   #in Hz
        up.paramSet.elim_vhf.Dpass = 0.05
        up.paramSet.elim_vhf.Dstop = 0.03

        # perform BPF
        s = DotMap()
        s.v = sig
        s.fs = fs

        b, a = signal.iirfilter(5, [2 * np.pi * lower_cutoff, 2 * np.pi * upper_cutoff], rs=60,
                                btype='band', analog=True, ftype='cheby2')

        bpf_sig = filtfilt(b, 1, s.v)

        return bpf_sig
def estimate_HR(sig: np.array, fs: int, up: DotMap, hr_past: int):
        """
        Heart Rate Estimation function estimate the heart rate according to the previous heart rate in given time window

        :param sig: array of signal with shape (N,) where N is the length of the signal
        :type sig: 1-d array
        :type fs: int
        :param up: setup up parameters of the algorithm
        :type up: DotMap
        :param hr_past: the average heart rate in the past in given time window
        :type hr_past: int

        :return: estimated heart rate, 1-d array.

        """

        # Estimate PSD
        blackman_window = np.blackman(len(sig))
        f, pxx = periodogram(sig,fs, blackman_window)
        ph = pxx
        fh = f

        # Extract HR
        if (hr_past / 60 < up.fl_hz) | (hr_past / 60 > up.fh_hz):
            rel_els = np.where((fh >= up.fl_hz) & (fh <= up.fh_hz))
        else:
            rel_els = np.where((fh >= hr_past / 60 * 0.5) & (fh <= hr_past / 60 * 1.4))

        rel_p = ph[rel_els]
        rel_f = fh[rel_els]
        max_el = np.where(rel_p==max(rel_p))
        hr = rel_f[max_el]*60
        hr = int(hr[0])

        return hr    

def estimate_deriv(sig: np.array):
        """
        Derivative Estimation function estimate derivative from highly filtered signal based on the
        General least-squares smoothing and differentiation by the convolution (Savitzky Golay) method

        :param sig: array of signal with shape (N,) where N is the length of the signal
        :type sig: 1-d array

        :return: derivative, 1-d array.

        """

        #Savitzky Golay
        deriv_no = 1
        win_size = 5
        deriv = savitzky_golay(sig, deriv_no, win_size)

        return deriv

def find_onsets(sig: np.array, fs: int, up: DotMap, peaks: np.array, med_hr: float):
        """
        This function finds the onsets of PPG sigal

        :param sig: array of signal with shape (N,) where N is the length of the signal
        :type sig: 1-d array
        :param fs: sampling frequency
        :type fs: int
        :param up: setup up parameters of the algorithm
        :type up: DotMap
        :param peaks: peaks of the signal
        :type peaks: 1-d array
        :param med_hr: median heart rate
        :type med_hr: float

        :return: onsets, 1-d array

        """

        Y1=def_bandpass(sig, fs, 0.9*up.fl_hz, 3*up.fh_hz)
        temp_oi0=find_peaks(-Y1,distance=med_hr*0.3)[0]

        null_indexes = np.where(temp_oi0<peaks[0])
        if len(null_indexes[0])!=0:
            if len(null_indexes[0])==1:
                onsets = [null_indexes[0][0]]
            else:
                onsets = [null_indexes[0][-1]]
        else:
            onsets = [peaks[0]-round(fs/50)]

        i=1
        while i < len(peaks):
            min_SUT=fs*0.12     # minimum Systolic Upslope Time 120 ms
            min_DT=fs*0.3       # minimum Diastolic Time 300 ms

            before_peak=temp_oi0 <peaks[i]
            after_last_onset=temp_oi0 > onsets[i - 1]
            SUT_time=peaks[i]-temp_oi0>min_SUT
            DT_time = temp_oi0-peaks[i-1]  > min_DT
            temp_oi1 = temp_oi0[np.where(before_peak * after_last_onset*SUT_time*DT_time)]
            if len(temp_oi1)>0:
                if len(temp_oi1) == 1:
                    onsets.append(temp_oi1[0])
                else:
                    onsets.append(temp_oi1[-1])
                i=i+1
            else:
                peaks = np.delete(peaks, i)

        return onsets,peaks

def savitzky_golay(sig: np.array, deriv_no: int, win_size: int):
        """
        This function estimate the Savitzky Golay derivative from highly filtered signal

        :param sig: array of signal with shape (N,) where N is the length of the signal
        :type sig: 1-d array
        :param deriv_no: number of derivative
        :type deriv_no: int
        :param win_size: size of window
        :type win_size: int

        :return: Savitzky Golay derivative, 1-d array.

        """

        ##assign coefficients
        # From: https: // en.wikipedia.org / wiki / Savitzky % E2 % 80 % 93 Golay_filter  # Tables_of_selected_convolution_coefficients
        # which are calculated from: A., Gorry(1990). "General least-squares smoothing and differentiation by the convolution (Savitzky?Golay) method".Analytical Chemistry. 62(6): 570?3. doi: 10.1021 / ac00205a007.

        if deriv_no==0:
            #smoothing
            if win_size == 5:
                coeffs = [-3, 12, 17, 12, -3]
                norm_factor = 35
            elif win_size == 7:
                coeffs = [-2, 3, 6, 7, 6, 3, -2]
                norm_factor = 21
            elif win_size == 9:
                coeffs = [-21, 14, 39, 54, 59, 54, 39, 14, -21]
                norm_factor = 231
            else:
                print('Can''t do this window size')
        elif deriv_no==1:
            # first derivative
            if win_size == 5:
                coeffs = range(-2,3)
                norm_factor = 10
            elif win_size == 7:
                coeffs = range(-3,4)
                norm_factor = 28
            elif win_size == 9:
                coeffs = range(-4,5)
                norm_factor = 60
            else:
                print('Can''t do this window size')
        elif deriv_no == 2:
            # second derivative
            if win_size == 5:
                coeffs = [2, -1, -2, -1, 2]
                norm_factor = 7
            elif win_size == 7:
                coeffs = [5, 0, -3, -4, -3, 0, 5]
                norm_factor = 42
            elif win_size == 9:
                coeffs = [28, 7, -8, -17, -20, -17, -8, 7, 28]
                norm_factor = 462
            else:
                print('Can''t do this window size')
        elif deriv_no == 3:
            # third derivative
            if win_size == 5:
                coeffs = [-1, 2, 0, -2, 1]
                norm_factor = 2
            elif win_size == 7:
                coeffs = [-1, 1, 1, 0, -1, -1, 1]
                norm_factor = 6
            elif win_size == 9:
                coeffs = [-14, 7, 13, 9, 0, -9, -13, -7, 14]
                norm_factor = 198
            else:
                print('Can''t do this window size')
        elif deriv_no == 4:
            # fourth derivative
            if win_size == 7:
                coeffs = [3, -7, 1, 6, 1, -7, 3]
                norm_factor = 11
            elif win_size == 9:
                coeffs = [14, -21, -11, 9, 18, 9, -11, -21, 14]
                norm_factor = 143
            else:
                print('Can''t do this window size')
        else:
            print('Can''t do this order of derivative')


        if deriv_no % 2 == 1:
            coeffs = -np.array(coeffs)

        A = [1, 0]
        filtered_sig = lfilter(coeffs, A, sig)
        # filtered_sig = filtfilt(coeffs, A, sig)
        s = len(sig)
        half_win_size = np.floor(win_size * 0.5)
        zero_pad=filtered_sig[win_size] * np.ones(int(half_win_size))
        sig_in=filtered_sig[win_size-1:s]
        sig_end=filtered_sig[s-1] * np.ones(int(half_win_size))
        deriv = [*zero_pad,*sig_in,*sig_end]
        deriv = deriv / np.array(norm_factor)

        return deriv

In [10]:
# from ppg_fiducials import get_peak_onset

_, sys_feet = get_peak_onset(ppg = ppg_filtered, fs = 55, peak_detector='PPGdet')

In [26]:
# time of the rpeaks in datetime
t_sysfeet = ppg.index.to_series().values[sys_feet]

# Go back to pandas Series
ppg = pd.Series(ppg_filtered, index = ppg.index)

# Plot
plt.figure(figsize = (19,8))
plt.subplot(2, 1, 1)
plt.plot(ppg)
plt.plot(t_sysfeet, ppg.loc[t_sysfeet], 'r*')
plt.xticks(fontsize = 19)
plt.yticks(fontsize = 19)
plt.ylabel("PPG (a.u.)", fontsize = 19)
plt.title("PPG and systolic feet", fontsize = 21)
plt.subplot(2, 1, 2, sharex = plt.subplot(2, 1, 1))
plt.plot(acc)
plt.xticks(fontsize = 19)
plt.yticks(fontsize = 19)
plt.ylabel("ACC (mg)", fontsize = 19)
plt.title("Accelerometer Data", fontsize = 21)
plt.show()

## Plot VeritySense acc together with Garmin ACC

In [21]:
GARMIN_ACC_PATH = Path(input_path("verity_sense_path"), "garmin_acc/")

garmin_acc = pd.read_parquet(GARMIN_ACC_PATH / "acc.parquet")

vs_offset_days = (garmin_acc.index[0] - ppg.index[0]).days
ppg.index = ppg.index + pd.Timedelta(days = vs_offset_days)
vs_offset_days = (garmin_acc.index[0] - acc.index[0]).days
acc.index = acc.index + pd.Timedelta(days = vs_offset_days)

In [24]:
# Plot
plt.figure(figsize = (19,8))
plt.subplot(2, 1, 1)
plt.plot(acc)
plt.xticks(fontsize = 19)
plt.yticks(fontsize = 19)
plt.title("Verity Sense ACC (mg)", fontsize = 19)
plt.subplot(2, 1, 2, sharex = plt.subplot(2, 1, 1))
plt.plot(garmin_acc)
plt.xticks(fontsize = 19)
plt.yticks(fontsize = 19)
plt.title("Garmin ACC (mg)", fontsize = 21)
plt.tight_layout()
plt.show()

In [29]:
offset

Timedelta('0 days 00:03:29.600000')

In [28]:
garmin_start_rotation =pd.Timestamp("2026-09-27 23:04:44.6")
vs_start_rotation = pd.Timestamp("2026-09-27 23:01:15")

offset = garmin_start_rotation - vs_start_rotation
# Adjust vs time to match garmin time
acc.index = acc.index + offset

plt.figure(figsize = (19,8))
plt.subplot(2, 1, 1)
plt.plot(acc)
plt.xticks(fontsize = 19)
plt.yticks(fontsize = 19)
plt.title("Verity Sense ACC (mg)", fontsize = 19)
plt.subplot(2, 1, 2, sharex = plt.subplot(2, 1, 1))
plt.plot(garmin_acc)
plt.xticks(fontsize = 19)
plt.yticks(fontsize = 19)
plt.title("Garmin ACC (mg)", fontsize = 21)
plt.tight_layout()
plt.show()

### Extract inter-beat interval time-series: RR intervals (from ECG) and PP intervals (from PPG)

In [11]:
rr = np.diff(t_rpeaks).astype('timedelta64[ns]').astype('float64') / 1000000000 # seconds
rr = np.insert(rr, 0, 0, axis = 0)
rr[0] = np.mean(rr[1:])

rr = pd.Series(rr, index = t_rpeaks)

pp = np.diff(t_sysfeet).astype('timedelta64[ns]').astype('float64') / 1000000000 # seconds
pp = np.insert(pp, 0, 0, axis = 0)
pp[0] = np.mean(pp[1:])

pp = pd.Series(pp, index = t_sysfeet)

plt.figure(figsize = (19,11))
plt.subplot(2, 1, 1)
plt.plot(60/rr)
plt.xticks(fontsize = 19)
plt.yticks(fontsize = 19)
plt.ylabel("RR intervals [s]", fontsize = 19)
plt.subplot(2, 1, 2, sharex = plt.subplot(2, 1, 1), sharey = plt.subplot(2, 1, 1))
plt.plot(60/pp)
plt.xticks(fontsize = 19)
plt.yticks(fontsize = 19)
plt.ylabel("PP intervals [s]", fontsize = 19)
plt.suptitle("Inter-beat intervals from ECG and PPG")
plt.tight_layout()

### Synchronize ECG and PPG 

From the previous plot we can see that RR intervals and PP intervals are not synchronized in time. This is because the internal clocks of the two devices are independent and not synchronized natively. 

How can we synchronize the two signals? --> Cross correlation function (CCF)

By shifting one series in relation to the other and calculating the dot-product at each point, we obtain the strength of the correlation at each shift

In [27]:
# manually select a clean portion of ecg and ppg
rr_for_synch = rr.loc[pd.Timestamp("2024-03-16 02:23:00"):pd.Timestamp("2024-03-16 02:40:30")]
pp_for_synch = pp.loc[pd.Timestamp("2024-03-16 02:23:00"):pd.Timestamp("2024-03-16 02:40:30")]    

plt.figure(figsize = (19,8))
plt.plot(rr_for_synch, label = "rr")
plt.plot(pp_for_synch, label = "pp")
plt.legend(fontsize = 19)

In [23]:
ecg.index[0]

Timestamp('2024-03-15 11:14:43.403294976')

In [17]:
from scipy import signal

In [49]:
def crosscorr(datax, datay, lag=0, wrap=False):
    
    return datay.corr(datax.shift(lag))

In [55]:
pp.index = pp.index - pd.Timedelta(seconds = 35)

In [53]:
rr_for_synch.to_pickle("rr_chatGPT.pkl")
pp_for_synch.to_pickle("pp_chatGPT.pkl")


In [57]:
plt.figure(figsize = (19,11))
plt.subplot(2, 1, 1)
plt.plot(rr)
plt.xticks(fontsize = 19)
plt.yticks(fontsize = 19)
plt.ylabel("RR intervals [s]", fontsize = 19)
plt.subplot(2, 1, 2, sharex = plt.subplot(2, 1, 1), sharey = plt.subplot(2, 1, 1))
plt.plot(pp)
plt.xticks(fontsize = 19)
plt.yticks(fontsize = 19)
plt.ylabel("PP intervals [s]", fontsize = 19)
plt.suptitle("Inter-beat intervals from ECG and PPG")
plt.tight_layout()

In [52]:
ccf = [crosscorr(rr_for_synch.reset_index(drop = True), pp_for_synch.reset_index(drop = True), lag) for lag in range(-40,40)]
offset = np.argmax(ccf) - 40
t_ccf = np.arange(-40, 40)

plt.figure()
plt.plot(t_ccf, ccf, linewidth=2.1)
plt.axvline(0,color='k',linestyle='--',label='Center', linewidth=1.6)
plt.axvline(offset,color='r',linestyle='--',label='Peak synchrony', linewidth=2.1)
plt.axhline(0, color='k', linestyle='--', linewidth=0.5)
plt.title(f'Offset = {offset} beats', fontsize = 20)
plt.xlabel('Time Shift (s)', fontsize = 20)
plt.ylabel('Correlation Coefficient', fontsize = 20)
plt.yticks(fontsize=20);
plt.xticks(fontsize=20);
plt.legend(['cross-correlation', 'center', 'peak synchrony'], fontsize=20)

In [ ]:
# Shift pp according to offset
pp_shifted = pp_for_synch.shift(offset)


## First algorithm for artifact detection: Kubios 

Paper: 
- A robust algorithm for heart rate variability time series artefact correction using novel beat classification. Lipponen & Tarvainen, 2019

Implementation:
- [https://neuropsychology.github.io/NeuroKit/_modules/neurokit2/signal/signal_fixpeaks.html#signal_fixpeaks](https://neuropsychology.github.io/NeuroKit/_modules/neurokit2/signal/signal_fixpeaks.html#signal_fixpeaks)

Key points: 
- It works on the inter-beat interval time-series, not on the raw data
- It's based on thresholds estimated from the distribution of consecutive differences of inter-beat intervals

In [12]:
artifacts_ecg, rpeaks_clean = nk.signal_fixpeaks(peaks=rpeaks, sampling_rate=130, iterative=False, show=False)
artifacts_ppg, sysfeet_clean = nk.signal_fixpeaks(peaks=sys_feet, sampling_rate=55, iterative=False, show=False)

artifacts_ecg

{'ectopic': [3,
  4,
  5,
  61,
  138,
  260,
  369,
  1622,
  2033,
  2043,
  2044,
  2067,
  2273,
  2505,
  2506,
  2527,
  2530,
  2534,
  2864,
  3146,
  3199,
  3724,
  3736,
  4096,
  4457,
  4464,
  4468,
  4603,
  4709,
  4756,
  5368,
  5369,
  5413,
  5417,
  5762,
  5875,
  5890,
  6081,
  6144,
  6234,
  6297,
  6354,
  6734,
  6751,
  7319,
  7454,
  7557,
  7978,
  8004,
  8017,
  8421,
  9242,
  9420,
  10162,
  10700,
  11201,
  11304,
  11444,
  11482,
  11921,
  12028,
  12053,
  12533,
  13616,
  13829,
  14041,
  14149,
  14529,
  14684,
  14762,
  15164,
  15203,
  15273,
  15308,
  15325,
  15524,
  15572,
  15573,
  15574,
  15575,
  15598,
  15599,
  15637,
  15641,
  15651,
  15667,
  15694,
  15953,
  15986,
  15987,
  16084,
  16123,
  16126,
  16198,
  16402,
  16712,
  16772,
  16967,
  17037,
  17212,
  17513,
  17514,
  17995,
  18805,
  19168,
  19250,
  19275,
  19309,
  19324,
  19994,
  20179,
  20190,
  20981,
  21449,
  22450,
  23989,
  24088,
  2

In [61]:
artifacts_ecg.keys()

dict_keys(['ectopic', 'missed', 'extra', 'longshort', 'method', 'rr', 'drrs', 'mrrs', 's12', 's22', 'c1', 'c2'])

In [62]:
plt.figure()
plt.plot(artifacts_ecg["drrs"])

In [13]:
rr_kubios = artifacts_ecg["rr"]
pp_kubios = artifacts_ppg["rr"]

artifacts_ecg_kubios = np.concatenate((artifacts_ecg["ectopic"], artifacts_ecg["missed"], artifacts_ecg["extra"], artifacts_ecg["longshort"]))
artifacts_ecg_kubios = np.sort(artifacts_ecg_kubios).astype(int)

artifacts_ppg_kubios = np.concatenate((artifacts_ppg["ectopic"], artifacts_ppg["missed"], artifacts_ppg["extra"], artifacts_ppg["longshort"]))
artifacts_ppg_kubios = np.sort(artifacts_ppg_kubios).astype(int)

t_artifacts_ecg_kubios = t_rpeaks[artifacts_ecg_kubios.astype(int)] 
t_artifacts_ppg_kubios = t_sysfeet[artifacts_ppg_kubios.astype(int)]

ecg_kubios = pd.DataFrame({"rr": rr_kubios})
ecg_kubios.index = t_rpeaks
ppg_kubios = pd.DataFrame({"pp": pp_kubios})
ppg_kubios.index = t_sysfeet

# 0 if no artifact, 1 if artifact
ecg_kubios["artifacts"] = 0
ecg_kubios["artifacts"][artifacts_ecg_kubios] = 1
ppg_kubios["artifacts"] = 0
ppg_kubios["artifacts"][artifacts_ppg_kubios] = 1

plt.figure()
plt.subplot(2,1,1)
plt.plot(ecg_kubios[ecg_kubios["artifacts"] == 0]["rr"], 'b-')
plt.plot(ecg_kubios[ecg_kubios["artifacts"] == 1]["rr"], 'r-*')
plt.subplot(2,1,2, sharex = plt.subplot(2,1,1))
plt.plot(ppg_kubios[ppg_kubios["artifacts"] == 0]["pp"], 'b-')
plt.plot(ppg_kubios[ppg_kubios["artifacts"] == 1]["pp"], 'r-*')

/var/folders/k1/8pz3zwd53cv4xvdj2pstv1440000gn/T/ipykernel_5852/3574821746.py:20: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  ecg_kubios["artifacts"][artifacts_ecg_kubios] = 1
/var/folders/k1/8pz3zwd53cv4xvdj2pstv1440000gn/T/ipykernel_5852